In [1]:
import time
import re
import logging
import yaml
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup


Создаем логгер с параметрами

In [2]:
with open('config_log.yaml', 'r') as f:
    config = yaml.safe_load(f)
if config['logging']['turn_on']:
    logging.basicConfig(filename=config['logging']['file'], level=config['logging']['level'], format='%(asctime)s (%(name)s) [%(levelname)s] - %(message)s')
    logger_scrap = logging.getLogger('steam_scraper')
else:
    logger_scrap = logging.getLogger('steam_scraper')
    logger_scrap.addHandler(logging.NullHandler())



Листаем сайт и забираем 20к карточек

In [3]:
options = webdriver.ChromeOptions()
driver = webdriver.Chrome(options=options)
logger_scrap.info("Начало парсинга со стима")
options.add_experimental_option("detach", True)
wait = WebDriverWait(driver, 10)

parsed = []
offset = 0
k = 12
all = 20005

driver.get('https://store.steampowered.com/games/?l=russian')
time.sleep(5)
logger_scrap.info("Успешно открыт сайт")

while len(parsed) < all:
    site = f'https://store.steampowered.com/games/?l=russian&offset={offset}'

    try:
        driver.get(site)
        wait.until(EC.presence_of_element_located((By.CLASS_NAME, 'LibraryAssetExpandedDisplay')))
        time.sleep(1)
        soup = BeautifulSoup(driver.page_source, 'html.parser')
        cards = soup.find_all('div', class_='LibraryAssetExpandedDisplay')
        parsed.extend(cards)
        if offset % 480 == 0:
            logger_scrap.info(f"Offset {offset} — на странице: {len(cards)}, итого: {len(parsed)}")
        offset += k

    except Exception as e:
        logger_scrap.error(f"Ошибка на offset {offset}: {e}")
        continue

driver.quit()
logger_scrap.info(f"Всего карточек: {len(parsed)}")


Парсим карточки

In [4]:
result = []
for card in parsed:
    try:
        img = card.find('img', class_='_2eQ4mkpf4IzUp1e9NnM2Wr')
        if img: 
            name = img['alt']
        else:
            name = None

        platform = card.find_all('span', title=True)
        platforms = [s['title'] for s in platform]
        platforms = ', '.join(platforms)
        if platforms:
            platforms = ', '.join(platforms)
        else:
            platforms = None

        date = card.find('div', class_='vCEpeeiHJkcIDdtTkRfjT')
        release_date = None
        if date:
            span = date.find('span')
            if span:
                release_date = span.text.strip()
            else:
                release_date = None

        tag = card.find_all('a', class_='WidgetTag')
        tags = [i.text.strip() for i in tag]
        if tags:
            tags = ', '.join(tags)
        else:
            tags = None

        ratings = card.find('div', class_='uEsfJ0VAuX37ItihZkK2J')
        if ratings:
            rating = ratings.text.strip()
        else:
            rating = None

        reviews = card.find('div', class_='yYag_VAd2NXLTrOBd-6mu')
        reviews_count = None
        if reviews and reviews['aria-label']:
            aria_label = reviews['aria-label']
            strs = re.search(r':\s*([\d\s\xa0]+)', aria_label)
            if strs:
                reviews_count = int(re.sub(r'[\s\xa0]', '', strs.group(1)))

        price = card.find('div', class_='_3j4dI1yA7cRfCvK8h406OB')
        total_price = None
        if price:
            strs = re.search(r'[\d.]+', price.text)
            if strs:
                total_price = float(strs.group())
            else:
                total_price = None

        result.append({'name': name, 'platforms': platforms, 'release_date': release_date, 'tags': tags, 'rating': rating, 'reviews_count': reviews_count,  'total_price': total_price,})

    except Exception as e:
        logger_scrap.error(f"Ошибка парсинга карточки {name}: {e}")
        continue

Сохраняем в датасет

In [5]:
df = pd.DataFrame(result)
logger_scrap.info(f"Итого игр: {len(df)}")
df.to_csv('steam_scrapped.csv', index=False, encoding='utf-8')
logger_scrap.info("Данные сохранены")

df.head()

,name,platforms,release_date,tags,rating,reviews_count,total_price
0,Counter-Strike 2,"W, i, n, d, o, w, s, ,, , L, i, n, u, x, , /...",21 авг. 2012 г.,"Шутер от первого лица, Шутер, Для нескольких и...",Очень положительные,2605647.0,NaN
1,Crimson Desert,"W, i, n, d, o, w, s, ,, , m, a, c, O, S",20 мар. 2026 г.,"Открытый мир, Экшен, Для одного игрока, Приклю...",В основном положительные,6417.0,69.99
2,Apex Legends™,"W, i, n, d, o, w, s",5 нояб. 2020 г.,"Бесплатная игра, Королевская битва, Для нескол...",В основном положительные,112372.0,NaN
3,Slay the Spire 2,"W, i, n, d, o, w, s, ,, , m, a, c, O, S, ,, ...",5 мар. 2026 г.,"Стратегия, Рогалик, Карточная игра, Построение...",Очень положительные,1288.0,12.49
4,Resident Evil 4,"W, i, n, d, o, w, s",24 мар. 2023 г.,"Хоррор, Экшен, Хоррор на выживание, Шутер от т...",Крайне положительные,7655.0,11.99


In [6]:
df.shape

(20016, 7)

In [7]:
df.head(5)

,name,platforms,release_date,tags,rating,reviews_count,total_price
0,Counter-Strike 2,"W, i, n, d, o, w, s, ,, , L, i, n, u, x, , /...",21 авг. 2012 г.,"Шутер от первого лица, Шутер, Для нескольких и...",Очень положительные,2605647.0,NaN
1,Crimson Desert,"W, i, n, d, o, w, s, ,, , m, a, c, O, S",20 мар. 2026 г.,"Открытый мир, Экшен, Для одного игрока, Приклю...",В основном положительные,6417.0,69.99
2,Apex Legends™,"W, i, n, d, o, w, s",5 нояб. 2020 г.,"Бесплатная игра, Королевская битва, Для нескол...",В основном положительные,112372.0,NaN
3,Slay the Spire 2,"W, i, n, d, o, w, s, ,, , m, a, c, O, S, ,, ...",5 мар. 2026 г.,"Стратегия, Рогалик, Карточная игра, Построение...",Очень положительные,1288.0,12.49
4,Resident Evil 4,"W, i, n, d, o, w, s",24 мар. 2023 г.,"Хоррор, Экшен, Хоррор на выживание, Шутер от т...",Крайне положительные,7655.0,11.99


In [8]:
df.isna().sum()

name              120
platforms         120
release_date        0
tags                0
rating           1207
reviews_count    1207
total_price      1121
dtype: int64

In [9]:
df['platforms']

0        W, i, n, d, o, w, s, ,,  , L, i, n, u, x,  , /...
1                 W, i, n, d, o, w, s, ,,  , m, a, c, O, S
2                                      W, i, n, d, o, w, s
3        W, i, n, d, o, w, s, ,,  , m, a, c, O, S, ,,  ...
4                                      W, i, n, d, o, w, s
                               ...                        
20011                                  W, i, n, d, o, w, s
20012    W, i, n, d, o, w, s, ,,  , m, a, c, O, S, ,,  ...
20013                                  W, i, n, d, o, w, s
20014                                  W, i, n, d, o, w, s
20015    W, i, n, d, o, w, s, ,,  , m, a, c, O, S, ,,  ...
Name: platforms, Length: 20016, dtype: str

In [10]:
df = df.drop(columns=['platforms'])

In [11]:
df

,name,release_date,tags,rating,reviews_count,total_price
0,Counter-Strike 2,21 авг. 2012 г.,"Шутер от первого лица, Шутер, Для нескольких и...",Очень положительные,2605647.0,NaN
1,Crimson Desert,20 мар. 2026 г.,"Открытый мир, Экшен, Для одного игрока, Приклю...",В основном положительные,6417.0,69.99
2,Apex Legends™,5 нояб. 2020 г.,"Бесплатная игра, Королевская битва, Для нескол...",В основном положительные,112372.0,NaN
3,Slay the Spire 2,5 мар. 2026 г.,"Стратегия, Рогалик, Карточная игра, Построение...",Очень положительные,1288.0,12.49
4,Resident Evil 4,24 мар. 2023 г.,"Хоррор, Экшен, Хоррор на выживание, Шутер от т...",Крайне положительные,7655.0,11.99
...,...,...,...,...,...,...
20011,Gray Protocol,23 февр. 2026 г.,"Приключение, Визуальная новелла, Глубокий сюже...",NaN,NaN,6.69
20012,Pizza Connection 3,22 мар. 2018 г.,"Стратегия, Симулятор, Менеджмент, Экономика, С...",Смешанные,661.0,10.49
20013,100 Africa Cats,1 окт. 2025 г.,"Казуальная игра, Бесплатная игра, Котики, Поис...",Положительные,18.0,1.39
20014,Urbex Awakening,5 дек. 2024 г.,"Симулятор ходьбы, Хоррор, Детектив, Исследован...",NaN,NaN,4.99


In [15]:
df = pd.read_csv('steam_scrapped.csv')
df = df.drop(columns=['platforms'])
df

,name,release_date,tags,rating,reviews_count,total_price
0,Counter-Strike 2,21 авг. 2012 г.,"Шутер от первого лица, Шутер, Для нескольких и...",Очень положительные,2605647.0,NaN
1,Crimson Desert,20 мар. 2026 г.,"Открытый мир, Экшен, Для одного игрока, Приклю...",В основном положительные,6417.0,69.99
2,Apex Legends™,5 нояб. 2020 г.,"Бесплатная игра, Королевская битва, Для нескол...",В основном положительные,112372.0,NaN
3,Slay the Spire 2,5 мар. 2026 г.,"Стратегия, Рогалик, Карточная игра, Построение...",Очень положительные,1288.0,12.49
4,Resident Evil 4,24 мар. 2023 г.,"Хоррор, Экшен, Хоррор на выживание, Шутер от т...",Крайне положительные,7655.0,11.99
...,...,...,...,...,...,...
20011,Gray Protocol,23 февр. 2026 г.,"Приключение, Визуальная новелла, Глубокий сюже...",NaN,NaN,6.69
20012,Pizza Connection 3,22 мар. 2018 г.,"Стратегия, Симулятор, Менеджмент, Экономика, С...",Смешанные,661.0,10.49
20013,100 Africa Cats,1 окт. 2025 г.,"Казуальная игра, Бесплатная игра, Котики, Поис...",Положительные,18.0,1.39
20014,Urbex Awakening,5 дек. 2024 г.,"Симулятор ходьбы, Хоррор, Детектив, Исследован...",NaN,NaN,4.99


In [16]:
df.isna().sum()

name              120
release_date       11
tags                0
rating           1207
reviews_count    1207
total_price      1121
dtype: int64

Пропуски в именах и датах - ошибки в парсинге, в рейтинге - отсутствие рейтинга, в цене - бесплатная игра


In [17]:
df.to_csv('steam_scrapped.csv', index=False, encoding='utf-8')
